In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType



In [0]:
%run ../config/config-env


In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

In [0]:
constructor_df = (
        spark.read.table(bronze_table)
)
display(constructor_df)

In [0]:
constructor_selected_df=  constructor_df.select(
    F.col('constructorId'), 
    F.col('name'), 
    F.col('nationality'), 
    F.col('ingestion_timestamp'), 
    F.col('source_file')
                            

)
display(constructor_selected_df)

In [0]:
constructor_standard_df = (
        constructor_selected_df
            .withColumnsRenamed({'constructorId':'constructor_Id', 'name':'constructor_name'})
            
    )
display(constructor_standard_df)

In [0]:
constructor_duplicated_df = constructor_standard_df.dropDuplicates(['constructor_id'])

In [0]:
constructor_final_df = (
            constructor_duplicated_df
                .withColumn('nationality', F.initcap(F.col('nationality')))
            
            )
display(constructor_final_df)

In [0]:
(
    constructor_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
spark.read.table(silver_table).display()